# **Practical 2**
## **Name** : Aryan Jawanjal

## **Roll no.** : A-35
## **Sem** : 7



In [ ]:
import pandas as pd
import numpy as np
import re

data = [
    ("D1", "Machine learning is used in healthcare and finance."),
    ("D2", "Python is a popular programming language for data science."),
    ("D3", "Artificial intelligence includes machine learning and deep learning."),
    ("D4", "Data science uses Python, statistics and machine learning."),
    ("D5", "Cloud computing provides storage, networking and virtual machines."),
    ("D6", "Cyber security protects computer systems from attacks and malware."),
    ("D7", "Big data analytics helps organizations make better decisions."),
    ("D8", "Deep learning is a branch of artificial intelligence."),
    ("D9", "Python supports machine learning, data analysis and visualization."),
    ("D10", "Database management systems store and retrieve information efficiently.")
]

doc_ids = [item[0] for item in data]
documents = [item[1] for item in data]

print(f"Loaded {len(doc_ids)} documents.")

Loaded 10 documents.


In [ ]:
def preprocess(text):
    return re.findall(r'\b\w+\b', text.lower())

vocabulary = set()
processed_docs = []

for doc in documents:
    words = preprocess(doc)
    processed_docs.append(words)
    vocabulary.update(words)

vocabulary = sorted(list(vocabulary))

print(f"Extracted {len(vocabulary)} unique terms.")

Extracted 55 unique terms.


In [ ]:
td_matrix = pd.DataFrame(0, index=vocabulary, columns=doc_ids)

for i, words in enumerate(processed_docs):
    doc_id = doc_ids[i]
    for word in words:
        td_matrix.at[word, doc_id] = 1

print("Term-Document Incidence Matrix (Sample):")
display(td_matrix.head(10))

Term-Document Incidence Matrix (Sample):


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
a,0,1,0,0,0,0,0,1,0,0
analysis,0,0,0,0,0,0,0,0,1,0
analytics,0,0,0,0,0,0,1,0,0,0
and,1,0,1,1,1,1,0,0,1,1
artificial,0,0,1,0,0,0,0,1,0,0
attacks,0,0,0,0,0,1,0,0,0,0
better,0,0,0,0,0,0,1,0,0,0
big,0,0,0,0,0,0,1,0,0,0
branch,0,0,0,0,0,0,0,1,0,0
cloud,0,0,0,0,1,0,0,0,0,0


In [ ]:
def boolean_retrieval(query, matrix):
    tokens = query.split()

    result_vector = None
    current_operator = None

    i = 0
    while i < len(tokens):
        token = tokens[i]

        if token in ["AND", "OR"]:
            current_operator = token
        elif token == "NOT":
            i += 1
            term = tokens[i].lower()

            if term in matrix.index:
                term_vector = ~matrix.loc[term].astype(bool)
            else:
                term_vector = pd.Series(True, index=matrix.columns)

            if result_vector is None:
                result_vector = term_vector
            elif current_operator == "AND":
                result_vector = result_vector & term_vector
            elif current_operator == "OR":
                result_vector = result_vector | term_vector

        else:
            term = token.lower()
            if term in matrix.index:
                term_vector = matrix.loc[term].astype(bool)
            else:
                term_vector = pd.Series(False, index=matrix.columns)

            if result_vector is None:
                result_vector = term_vector
            elif current_operator == "AND":
                result_vector = result_vector & term_vector
            elif current_operator == "OR":
                result_vector = result_vector | term_vector

        i += 1

    if result_vector is not None:
        matched_docs = result_vector[result_vector].index.tolist()
        return matched_docs
    return []

In [ ]:
queries = [
    "machine AND learning",
    "python OR data",
    "machine AND NOT finance",
    "deep AND learning AND artificial"
]

for q in queries:
    matched = boolean_retrieval(q, td_matrix)
    print(f"Query: '{q}'")
    print(f"Matched Documents: {matched}")
    print("-" * 40)

Query: 'machine AND learning'
Matched Documents: ['D1', 'D3', 'D4', 'D9']
----------------------------------------
Query: 'python OR data'
Matched Documents: ['D2', 'D4', 'D7', 'D9']
----------------------------------------
Query: 'machine AND NOT finance'
Matched Documents: ['D3', 'D4', 'D9']
----------------------------------------
Query: 'deep AND learning AND artificial'
Matched Documents: ['D3', 'D8']
----------------------------------------
